# Amazon Product Reviews — Customer Behavior Analysis

In [ ]:
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## Load Data

In [1]:
df = pd.read_csv('../Datasets/Amazon_Product_Review.csv', encoding='latin1')
print(f'Shape: {df.shape}')
df.head(3)

NameError: name 'pd' is not defined

In [ ]:
pd.DataFrame({
    'dtype'  : df.dtypes,
    'nulls'  : df.isnull().sum(),
    'null_%' : (df.isnull().mean() * 100).round(1),
    'unique' : df.nunique()
})

## Data Cleaning

In [ ]:
# Drop columns that are 100% null or irrelevant to behaviour analysis
df.drop(columns=[
    'reviews.userCity', 'reviews.userProvince',
    'sizes', 'ean', 'upc', 'manufacturerNumber', 'colors', 'dimension'
], inplace=True)
print('Remaining columns:', list(df.columns))

In [ ]:
# Parse dates
for col in ['reviews.date', 'dateAdded', 'dateUpdated']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

df['review_year']  = df['reviews.date'].dt.year
df['review_month'] = df['reviews.date'].dt.month
print('Review date range:', df['reviews.date'].min().date(), '—', df['reviews.date'].max().date())

In [ ]:
# Extract USD price, sale flag, and currencies from the embedded JSON
def extract_usd_price(raw):
    try:
        entries = json.loads(raw)
        usd = [e for e in entries if e.get('currency') == 'USD']
        return (usd or entries)[0].get('amountMax', np.nan)
    except:
        return np.nan

def has_sale(raw):
    try:
        return any(str(e.get('isSale', 'false')).lower() == 'true' for e in json.loads(raw))
    except:
        return False

def extract_currencies(raw):
    try:
        return ', '.join(sorted({e.get('currency') for e in json.loads(raw) if e.get('currency')}))
    except:
        return np.nan

df['price_usd']  = df['prices'].apply(extract_usd_price)
df['on_sale']    = df['prices'].apply(has_sale)
df['currencies'] = df['prices'].apply(extract_currencies)
df['price_usd'].describe().round(2)

In [ ]:
# Clean the noisy categories field — pull the first meaningful token
noise = {'mazon.co.uk', 'categories', 'see more', '', 'electronics features', 'consumer electronics'}

def clean_categories(raw):
    if not isinstance(raw, str):
        return 'Unknown'
    parts = [p.strip() for p in raw.split(',')]
    parts = [p for p in parts if p.lower() not in noise and not p.lower().startswith('see more')]
    return parts[0] if parts else 'Unknown'

df['primary_category'] = df['categories'].apply(clean_categories)
df['primary_category'].value_counts().head(10)

In [ ]:
# Derive device type from product name
def get_device(name):
    n = str(name).lower()
    if 'echo' in n:                       return 'Echo'
    elif 'fire tv' in n:                  return 'Fire TV'
    elif 'fire hd' in n or 'hdx' in n:   return 'Fire Tablet'
    elif 'kindle' in n:                   return 'Kindle'
    elif 'tap' in n:                      return 'Amazon Tap'
    elif 'fire' in n:                     return 'Fire'
    return 'Accessory'

df['device_type'] = df['name'].apply(get_device)
df['device_type'].value_counts()

In [ ]:
# Clean review text — strip HTML, non-ASCII, and extra whitespace
def clean_text(txt):
    if not isinstance(txt, str): return ''
    txt = re.sub(r'<[^>]+>', ' ', txt)
    txt = re.sub(r'[^\x00-\x7F]+', ' ', txt)
    return re.sub(r'\s+', ' ', txt).strip()

df['reviews.text']  = df['reviews.text'].apply(clean_text)
df['reviews.title'] = df['reviews.title'].apply(clean_text)

# Fill missing ratings with per-product median, then overall median
df['reviews.rating'] = df.groupby('name')['reviews.rating'].transform(lambda x: x.fillna(x.median()))
df['reviews.rating'].fillna(df['reviews.rating'].median(), inplace=True)

df['reviews.username'].fillna('Anonymous', inplace=True)
df['reviews.title'].fillna('No Title', inplace=True)
df['reviews.doRecommend'] = df['reviews.doRecommend'].map({True: True, False: False, 'TRUE': True, 'FALSE': False})

print('Nulls remaining:')
remaining = df.isnull().sum()
print(remaining[remaining > 0])

In [ ]:
# Sentiment labels
df['sentiment'] = df['reviews.rating'].apply(
    lambda r: 'Positive' if r >= 4 else ('Neutral' if r == 3 else 'Negative')
)

pos_kw = ['love', 'great', 'excellent', 'amazing', 'perfect', 'awesome', 'best']
neg_kw = ['terrible', 'awful', 'worst', 'horrible', 'broken', 'defective', 'disappointed']

def keyword_sentiment(text):
    t = text.lower()
    pos = any(w in t for w in pos_kw)
    neg = any(w in t for w in neg_kw)
    if pos and not neg:  return 'Positive'
    if neg and not pos:  return 'Negative'
    return 'Neutral'

df['text_sentiment'] = df['reviews.text'].apply(keyword_sentiment)
print(df['sentiment'].value_counts())
print(df['text_sentiment'].value_counts())

## Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

rc = df['reviews.rating'].value_counts().sort_index()
axes[0].bar(rc.index.astype(int), rc.values, color=sns.color_palette('muted', 5), edgecolor='white')
axes[0].set(title='Rating Distribution', xlabel='Stars', ylabel='Reviews')
axes[0].set_xticks([1, 2, 3, 4, 5])

sc = df['sentiment'].value_counts()
axes[1].pie(sc, labels=sc.index, colors=['#4CAF50','#FFC107','#F44336'], autopct='%1.1f%%', startangle=140)
axes[1].set_title('Sentiment Breakdown')

plt.tight_layout()
plt.savefig('../outputs/charts/rating_distribution.png', dpi=150)
plt.show()

## Average Ratings by Category

In [ ]:
cat_stats = df.groupby('primary_category').agg(
    avg_rating   = ('reviews.rating', 'mean'),
    review_count = ('reviews.rating', 'count')
).sort_values('avg_rating', ascending=False).reset_index().round(2)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(cat_stats['primary_category'], cat_stats['avg_rating'],
               color=sns.color_palette('Blues_r', len(cat_stats)))
ax.set(xlabel='Average Rating', title='Average Rating by Category', xlim=(0, 5.8))
for bar, val in zip(bars, cat_stats['avg_rating']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/charts/avg_rating_by_category.png', dpi=150)
plt.show()
cat_stats

## Most Reviewed Products

In [ ]:
top_products = df.groupby('name').agg(
    review_count = ('reviews.text', 'count'),
    avg_rating   = ('reviews.rating', 'mean'),
    avg_price    = ('price_usd', 'mean')
).sort_values('review_count', ascending=False).head(15).reset_index().round(2)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_products['name'][::-1], top_products['review_count'][::-1],
        color=sns.color_palette('viridis', 15))
ax.set(xlabel='Number of Reviews', title='Top 15 Most Reviewed Products')
plt.tight_layout()
plt.savefig('../outputs/charts/top_products.png', dpi=150)
plt.show()
top_products

## Customer Shopping Frequency

In [ ]:
user_freq = df.groupby('reviews.username').agg(
    purchase_count   = ('name', 'count'),
    unique_products  = ('name', 'nunique'),
    avg_rating_given = ('reviews.rating', 'mean'),
    avg_spend        = ('price_usd', 'mean')
).reset_index()
user_freq.columns = ['username', 'purchase_count', 'unique_products', 'avg_rating_given', 'avg_spend']
user_freq = user_freq.round(2)

print('Frequency summary:')
print(user_freq['purchase_count'].describe().round(2))
print()
user_freq.sort_values('purchase_count', ascending=False).head(10)

In [ ]:
def freq_segment(n):
    if n == 1:    return 'One-time'
    elif n <= 3:  return 'Occasional (2-3)'
    elif n <= 7:  return 'Regular (4-7)'
    return 'Power Reviewer (8+)'

user_freq['freq_segment'] = user_freq['purchase_count'].apply(freq_segment)

sc = user_freq['freq_segment'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(sc.index, sc.values, color=sns.color_palette('Set2', 4))
ax.set(title='Shopping Frequency Segments', ylabel='Customers')
plt.tight_layout()
plt.savefig('../outputs/charts/frequency_segments.png', dpi=150)
plt.show()

## Preferred Shopping Categories

In [ ]:
cat_pref = df.groupby(['reviews.username', 'primary_category']).size().reset_index(name='count')
top_cat_per_user = (
    cat_pref.sort_values('count', ascending=False)
            .drop_duplicates('reviews.username')
            .rename(columns={'primary_category': 'preferred_category'})
)
print(top_cat_per_user['preferred_category'].value_counts())

fig = px.treemap(
    df.groupby('primary_category').size().reset_index(name='reviews'),
    path=['primary_category'], values='reviews', title='Review Volume by Category'
)
fig.write_html('../outputs/charts/category_treemap.html')
fig.show()

## Payment Preferences

In [ ]:
currency_counts = df['currencies'].dropna().str.split(', ').explode().value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(currency_counts.index, currency_counts.values, color=sns.color_palette('pastel', len(currency_counts)))
axes[0].set(title='Products by Currency / Market', xlabel='Currency')

sale_counts = df['on_sale'].value_counts()
axes[1].pie(sale_counts, labels=['Full Price','On Sale'][:len(sale_counts)],
            autopct='%1.1f%%', colors=['#42A5F5','#EF5350'])
axes[1].set_title('Full Price vs On Sale')

plt.tight_layout()
plt.savefig('../outputs/charts/payment_preferences.png', dpi=150)
plt.show()

## Device Usage Patterns

In [ ]:
device_stats = df.groupby('device_type').agg(
    review_count = ('reviews.text', 'count'),
    avg_rating   = ('reviews.rating', 'mean'),
    avg_price    = ('price_usd', 'mean'),
    positive_pct = ('sentiment', lambda x: (x == 'Positive').mean() * 100)
).reset_index().sort_values('review_count', ascending=False).round(2)

print(device_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col, title in zip(axes, ['review_count','avg_rating'], ['Review Count','Avg Rating']):
    ax.bar(device_stats['device_type'], device_stats[col], color=sns.color_palette('tab10', len(device_stats)))
    ax.set(title=f'{title} by Device Type')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../outputs/charts/device_usage.png', dpi=150)
plt.show()

## Device Type vs Purchase Frequency

> Age and Gender columns are not present in this dataset. Device type is used as the primary behavioural proxy.

In [ ]:
device_freq = df.merge(
    user_freq[['username','purchase_count','avg_rating_given']],
    left_on='reviews.username', right_on='username', how='left'
)
device_freq_grouped = device_freq.groupby('device_type').agg(
    avg_purchase_freq    = ('purchase_count', 'mean'),
    avg_rating_behaviour = ('avg_rating_given', 'mean')
).reset_index().round(2)

print(device_freq_grouped.to_string(index=False))

fig = px.scatter(device_freq_grouped, x='avg_purchase_freq', y='avg_rating_behaviour',
                 text='device_type', size=[60]*len(device_freq_grouped),
                 title='Device Type: Purchase Frequency vs Rating Generosity')
fig.write_html('../outputs/charts/device_freq_vs_rating.html')
fig.show()

## Customer Segmentation

In [ ]:
user_device = (
    df.groupby('reviews.username')['device_type']
      .agg(lambda x: x.value_counts().index[0])
      .reset_index(name='fav_device')
)

user_segments = (
    user_freq
    .merge(top_cat_per_user[['reviews.username','preferred_category']], on='reviews.username', how='left')
    .merge(user_device, on='reviews.username', how='left')
)

def assign_segment(row):
    if row['purchase_count'] >= 4 and row['avg_spend'] >= 80:   return 'High-Value Regular'
    if row['purchase_count'] >= 4 and row['avg_spend'] < 80:    return 'Frequent Budget Buyer'
    if row['purchase_count'] <= 2 and row['avg_rating_given'] >= 4.5: return 'Satisfied Casual'
    if row['avg_rating_given'] <= 2.5:                          return 'Dissatisfied Shopper'
    return 'Average Shopper'

user_segments['customer_segment'] = user_segments.apply(assign_segment, axis=1)
print(user_segments['customer_segment'].value_counts())

sc = user_segments['customer_segment'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(sc.index, sc.values, color=sns.color_palette('Set1', len(sc)))
ax.set(title='Customer Segments', ylabel='Customers')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('../outputs/charts/customer_segments.png', dpi=150)
plt.show()

## Products with Inconsistent Ratings

In [ ]:
rc = df.groupby('name')['reviews.rating'].agg(mean='mean', std='std', count='count').reset_index()
inconsistent = rc[(rc['count'] >= 5) & (rc['std'] > 1.5)].sort_values('std', ascending=False).round(2)

print(f'Inconsistent products: {len(inconsistent)}')

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(inconsistent['name'][:15][::-1], inconsistent['std'][:15][::-1], color='#EF5350')
ax.set(xlabel='Rating Std Dev', title='Products with Most Inconsistent Ratings (std > 1.5)')
plt.tight_layout()
plt.savefig('../outputs/charts/inconsistent_ratings.png', dpi=150)
plt.show()
inconsistent

## Recommended vs Not Recommended

In [ ]:
rec_df = df.dropna(subset=['reviews.doRecommend'])
print(f'Reviews with flag: {len(rec_df)} ({len(rec_df)/len(df)*100:.1f}%)')

rec_stats = rec_df.groupby('reviews.doRecommend').agg(
    count       = ('reviews.rating', 'count'),
    avg_rating  = ('reviews.rating', 'mean'),
    avg_helpful = ('reviews.numHelpful', 'mean')
).reset_index().round(2)
rec_stats['reviews.doRecommend'] = rec_stats['reviews.doRecommend'].map({True: 'Recommends', False: 'Does Not'})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col, title in zip(axes, ['avg_rating','avg_helpful'], ['Avg Rating','Avg Helpful Votes']):
    ax.bar(rec_stats['reviews.doRecommend'], rec_stats[col], color=['#4CAF50','#F44336'])
    ax.set(title=f'{title}: Recommends vs Does Not')
plt.tight_layout()
plt.savefig('../outputs/charts/recommend_comparison.png', dpi=150)
plt.show()
rec_stats

## Customer Personas

In [ ]:
persona_profile = user_segments.groupby('customer_segment').agg(
    count        = ('username', 'count'),
    avg_purchases= ('purchase_count', 'mean'),
    avg_spend    = ('avg_spend', 'mean'),
    avg_rating   = ('avg_rating_given', 'mean'),
    top_device   = ('fav_device',      lambda x: x.value_counts().index[0] if len(x) else 'N/A'),
    top_category = ('preferred_category', lambda x: x.value_counts().index[0] if len(x) else 'N/A')
).reset_index().round(2)
persona_profile

In [ ]:
personas = {
    'High-Value Regular'    : ('Frequent buyers spending $80+. Loyal to premium devices.',
                               'Upsell accessories. Prime-exclusive bundles and early access.'),
    'Frequent Budget Buyer' : ('4+ purchases but under $80 avg. Attracted to deals.',
                               'Promote sale events, bundles, and certified refurbished options.'),
    'Satisfied Casual'      : ('1-2 purchases, very high ratings. Deliberate buyer.',
                               'Use top reviews as social proof. Re-engage with recommendations.'),
    'Dissatisfied Shopper'  : ('Consistently low ratings. Vocal and detailed in feedback.',
                               'Proactive support outreach. Post-purchase satisfaction surveys.'),
    'Average Shopper'       : ('Mid-range on all metrics. Moderate engagement.',
                               'Email nudges and wishlist reminders to move them up.'),
}

for segment, (desc, action) in personas.items():
    print(f'\n{segment}')
    print(f'  Who    : {desc}')
    print(f'  Action : {action}')

## Marketing Insights

In [ ]:
total   = user_segments.shape[0]
hv      = (user_segments['customer_segment'] == 'High-Value Regular').sum()
dis     = (user_segments['customer_segment'] == 'Dissatisfied Shopper').sum()
td      = device_stats.iloc[0]
tc      = cat_stats.iloc[0]
mr      = top_products.iloc[0]
pct_pos = (df['sentiment'] == 'Positive').mean() * 100
pct_neg = (df['sentiment'] == 'Negative').mean() * 100

insights = [
    ('Sentiment',        f'{pct_pos:.1f}% positive, {pct_neg:.1f}% negative.',
                         f'Address the {pct_neg:.1f}% negative reviews to reduce churn.'),
    ('Top Device',       f'{td["device_type"]} — {td["avg_rating"]:.2f}/5 stars, {td["positive_pct"]:.1f}% positive.',
                         'Anchor campaigns here. Strongest base for cross-sell.'),
    ('Top Category',     f'"{tc["primary_category"]}" — {tc["review_count"]} reviews, {tc["avg_rating"]:.2f} avg.',
                         'Feature on homepage and email campaigns.'),
    ('High-Value',       f'{hv}/{total} customers ({hv/total*100:.1f}%) drive outsized engagement.',
                         'Create a loyalty tier to retain them before competitors do.'),
    ('Dissatisfied',     f'{dis} customers consistently rate products low.',
                         'Trigger support email post-purchase. 30-day satisfaction guarantee.'),
    ('Best Seller',      f'"{mr["name"]}" — {int(mr["review_count"])} reviews, {mr["avg_rating"]:.2f} stars.',
                         'Badge it as Best Seller. Use review volume as social proof in ads.'),
    ('Sale Sensitivity', f'{df["on_sale"].mean()*100:.1f}% products were on sale.',
                         'Time promotions around peak review months.'),
    ('One-time Buyers',  'Largest frequency segment is one-time purchasers.',
                         '30-day re-engagement email with complementary product picks.'),
]

print('=== MARKETING INSIGHTS ===\n')
for i, (title, finding, action) in enumerate(insights, 1):
    print(f'{i}. {title}')
    print(f'   Finding : {finding}')
    print(f'   Action  : {action}\n')

## Save Cleaned Data

In [ ]:
df.to_csv('../Datasets/Amazon_Product_Review_Cleaned.csv', index=False)
user_segments.to_csv('../Datasets/Customer_Segments.csv', index=False)

print('Saved: Datasets/Amazon_Product_Review_Cleaned.csv')
print('Saved: Datasets/Customer_Segments.csv')
print(f'Final shape: {df.shape}')